# 🚀 Google Colab Option B: Decoupled Harvester & Dedicated Trainer
### Kiến Trúc Tách Biệt: Cào Đẩy Dữ Liệu Lên GDrive Trước (Job 1) & Huấn Luyện Batch Tập Trung (Job 2)

Notebook này được thiết kế theo **Option B** chuẩn hoá nghiên cứu khoa học:
- 📂 **Job 1 (Harvester 4h)**: Cào dữ liệu người thật từ Threads & Wikimedia Commons, tạo mẫu đối chứng và **đẩy thẳng 100% lên Google Drive** định kỳ mỗi 4 tiếng. Hoàn toàn không tốn GPU quota của Colab.
- 🧠 **Job 2 (Dedicated Batch Trainer)**: Nạp GPU T4/V100, tự động gom các batch từ Google Drive về bộ nhớ đệm nhanh NVMe SSD (`/content/train_cache`), huấn luyện Model 1 (ViT) & Model 2 (Biometrics) với tốc độ tối đa không bị nghẽn mạng Google Drive FUSE.
- 🧹 **Công Cụ Bổ Trợ (Audit & Purge)**: Dọn sạch mọi mẫu tranh hoạt hình hoặc ảnh khuôn mặt AI cũ còn sót lại trên Drive.

--- 
## 📁 Bước 1: Kết Nối Google Drive (Mount Drive)

In [ ]:
#@title 📁 Kết Nối Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
print('✅ [Google Drive] Đã kết nối thành công!')

--- 
## 💓 Bước 2: Kích Hoạt Anti-Idle Heartbeat (Chống Ngắt Kết Nối Colab Free)

In [ ]:
#@title 💓 Kích Hoạt Anti-Idle JavaScript Heartbeat
import IPython.display as display

display.display(display.Javascript('''
function KeepAlive() {
    console.log("[Colab Free Heartbeat] Giữ kết nối active - " + new Date().toLocaleTimeString());
    let connectBtn = document.querySelector("colab-connect-button")?.shadowRoot?.querySelector("#connect");
    if (connectBtn) {
        connectBtn.click();
    }
}
setInterval(KeepAlive, 60000);
'''))

print("✅ Anti-Idle Heartbeat đã kích hoạt ngầm (Gửi tín hiệu mỗi 60s để giữ tab sống)!")

--- 
## 📦 Bước 3: Cập Nhật Mã Nguồn & Dependencies

In [ ]:
#@title 📦 Clone / Pull Repository & Dependencies
import os

if not os.path.exists('/content/DeepFake-Detection-Agent'):
    !git clone https://github.com/ThangCao106/DeepFake-Detection-Agent.git /content/DeepFake-Detection-Agent

%cd /content/DeepFake-Detection-Agent
!git checkout main
!git pull origin main
!pip install -q -r requirements.txt
print('✅ Đã nạp mã nguồn và cài đặt dependencies thành công!')

--- 
## 🕷️ JOB 1: Harvester 4h/Lần (Chỉ Cào & Đẩy 100% Dữ Liệu Lên GDrive)
> **Mục đích**: Chạy định kỳ thu thập ảnh thật từ Threads & Wikimedia Commons, sinh biến thể đối chứng và đẩy lên Google Drive theo thư mục giờ `batch_HHhMM`. Không bật train để tiết kiệm 100% GPU quota.

In [ ]:
#@title 🚀 Chạy Job 1: Harvester 4h (Cào & Sync GDrive)
%cd /content/DeepFake-Detection-Agent
!git pull origin main

IMAGES_CRAWL = 25 #@param {type:"integer"}
VIDEOS_CRAWL = 2 #@param {type:"integer"}
AUDIOS_CRAWL = 2 #@param {type:"integer"}
LOOP_INTERVAL = "4h" #@param ["4h", "1h", "30m", "none"]

cmd = f"python colab_sync_to_gdrive.py --images {IMAGES_CRAWL} --videos {VIDEOS_CRAWL} --audios {AUDIOS_CRAWL}"
if LOOP_INTERVAL != "none":
    cmd += f" --loop {LOOP_INTERVAL}"

!{cmd}

--- 
## 🧠 JOB 2: Dedicated Batch Trainer (Huấn Luyện Tập Trung Từ Dữ Liệu GDrive)
> **Mục đích**: Bật GPU (T4/V100/A100), tự động gom các batch đã tích lũy trên Google Drive về bộ nhớ đệm nhanh NVMe SSD (`/content/train_cache`), huấn luyện Model 1 & 2 với tốc độ cao nhất (không nghẽn FUSE) và lưu Checkpoints về Google Drive.

In [ ]:
#@title 🎯 Chạy Job 2: Dedicated Batch Trainer
%cd /content/DeepFake-Detection-Agent
!git pull origin main

TARGET_DATE = "2026-09-29" #@param {type:"string"} ["2026-09-29", "all"]
EPOCHS = 3 #@param {type:"integer"}
BATCH_SIZE = 16 #@param {type:"integer"}
LEARNING_RATE = 0.00001 #@param {type:"number"}
CACHE_TO_LOCAL = True #@param {type:"boolean"}

cmd = f"python colab_training/run_batch_trainer.py --date {TARGET_DATE} --epochs {EPOCHS} --batch-size {BATCH_SIZE} --lr {LEARNING_RATE}"
if not CACHE_TO_LOCAL:
    cmd += " --no-cache"

!{cmd}

--- 
## 🧹 CÔNG CỤ PHỤ TRỢ: Audit & Purge Dữ Liệu Cũ Trên Google Drive
> **Mục đích**: Quét sạch các ảnh hoạt hình hoặc 5 khuôn mặt AI cũ trong các thư mục ngày trên Google Drive và bù đắp bằng ảnh chụp người thật ngoài đời thực từ Wikimedia Commons.

In [ ]:
#@title 🧹 Quét & Thanh Lọc Dữ Liệu Hạt Giống AI Cũ (Purge & Replenish)
%cd /content/DeepFake-Detection-Agent
!git pull origin main

PURGE_DATE = "2026-09-29" #@param {type:"string"}
!python scripts/purge_synthetic_drawings.py --date {PURGE_DATE} --drive-mount /content/drive/MyDrive/Thread_Data